# Per-gene fine-mapping against a pre-built QtlDataset

## Description

For a single study's `QtlDataset` RDS, run `pecotmr::fineMappingPipeline(methods = "susie", ...)` per gene. Gene-level parallelization is via SoS task fan-out: each task loads the same RDS and fine-maps one gene. No pecotmr-internal accessors are called from this notebook — the wrapper hands the QtlDataset and a single `--gene-id` to the pipeline and writes one RDS per gene.

## Inputs

- `--qtl-dataset` &mdash; path to the QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes` &mdash; explicit gene-ID list (space-separated CLI args).
- `--cis-window` &mdash; bp window around each trait's TSS for variant selection. Default 1,000,000.
- `--coverage` &mdash; SuSiE credible-set coverage. Default 0.95.

## Output

- `{cwd}/{study}.{gene}.finemap.rds` per gene.


## Example

```bash
sos run pipeline/fine_mapping.ipynb fine_mapping \
    --cwd output \
    --study ROSMAP_DLPFC \
    --qtl-dataset output/ROSMAP_DLPFC.qtl_dataset.rds \
    --genes ENSG00000060237 ENSG00000234593 ENSG00000168487 \
    --cis-window 1000000
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: qtl_dataset = path
# Explicit gene list; for large fan-outs use the standard SoS @file.txt
# CLI sugar (one gene per line in the file).
parameter: genes = []
parameter: cis_window = 1000000
parameter: coverage = 0.95
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[fine_mapping]
# One task per gene; every task loads the same QtlDataset RDS once and
# runs fineMappingPipeline against one traitId.
input: qtl_dataset, for_each = 'genes'
output: f"{cwd}/{study}.{_genes}.finemap.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${path("code/script/pecotmr_integration/fine_mapping.R", absolute=True)} \
        --qtl-dataset ${_input} \
        --gene-id ${_genes} \
        --cis-window ${cis_window} \
        --coverage ${coverage} \
        --output ${_output}
